# Encontro 10 — Integração, orquestração e memória

**Tópicos Especiais em IA · Agentes Inteligentes · IFES Serra · 2026/2**

Este é o lab do encontro, e ele roda **inteiro aqui no Colab**.

> **Mas repare numa coisa, porque é o assunto de hoje:** o código deste lab **não está
> neste notebook**. Ele está em **arquivos `.py`**, organizados num pacote chamado
> `vertice/`, e o que você vai rodar aqui são **comandos que executam esses arquivos**.
>
> É a primeira vez no semestre que isso acontece, e é de propósito. O projeto cresceu
> além do que uma célula sustenta.

A migração para o VS Code, com `clone`, `commit` e `push`, está no
**`GUIA_VSCODE.md`** e é tarefa para fazer com calma, fora da pressão da aula.

## 1 · Trazer o lab para cá

Duas formas. Se o professor publicou o repositório, cole a URL abaixo. Se não, deixe
vazio e o Colab vai pedir o arquivo `lab-enc10-vertice.zip`.

In [ ]:
REPO_DO_LAB = ""     # <- cole aqui a URL do repositorio publico do lab, ou deixe vazio

import os, shutil, zipfile, pathlib

PASTA = "/content/lab-enc10"
shutil.rmtree(PASTA, ignore_errors=True)

if REPO_DO_LAB.strip():
    !git clone -q {REPO_DO_LAB} {PASTA}
else:
    from google.colab import files
    print("Escolha o arquivo lab-enc10-vertice.zip")
    enviado = files.upload()
    nome = list(enviado)[0]
    os.makedirs(PASTA, exist_ok=True)
    with zipfile.ZipFile(nome) as z:
        z.extractall(PASTA)

# se o zip ou o repo criou uma pasta a mais, entra nela
if not pathlib.Path(PASTA, "testes.py").exists():
    for sub in pathlib.Path(PASTA).iterdir():
        if sub.is_dir() and (sub / "testes.py").exists():
            PASTA = str(sub)
            break

os.chdir(PASTA)
print("estou em:", os.getcwd())
print()
!find . -name "*.py" -not -path "*/.*" | sort

**Olhe a lista acima.** Isto é um **pacote Python**: uma pasta com `__init__.py` dentro,
e módulos que se importam entre si. Cada arquivo tem uma responsabilidade:

| Arquivo | Responsabilidade |
|---|---|
| `vertice/dominio.py` | o mundo sintético e as seis consultas |
| `vertice/casos.py` | os chamados e a verdade de terra |
| `vertice/config.py` | chave, modelo e preços |
| `vertice/artesanal.py` | o laço do Encontro 3 |
| `vertice/framework.py` | o mesmo agente no Agent Framework |
| `vertice/medicao.py` | consultas, tokens, dólares e a tabela |

**Era tudo isso que cabia numa célula até o Encontro 9.** Não cabe mais.

## 2 · Instalar

As versões fixadas do `requirements.txt`. **Não instale o meta-pacote `agent-framework`**:
ele arrasta 343 pacotes; os dois que importam fecham em 24.

Se o Colab pedir para reiniciar a sessão, reinicie e **rode a célula 1 de novo**.

In [ ]:
%pip install -q -r requirements.txt
print("pronto. Se aparecer aviso de reiniciar a sessao, reinicie e volte a celula 1.")

## 3 · A chave

Continua vindo dos **Secrets do Colab**, como em todos os encontros desde o 2.

A célula lê o segredo e o coloca no **ambiente**, porque as células seguintes executam os
arquivos `.py` em outro processo, e esse processo não enxerga os *Secrets* direto.

**Nada é escrito em disco.** No VS Code, o mesmo módulo lê de um arquivo `.env` — e o
`config.py` procura nos dois lugares, sem mudar uma linha do agente.

In [ ]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
print("chave carregada, termina em:", os.environ["GROQ_API_KEY"][-4:])
print()
print("Chave individual, nunca compartilhada: os limites do provedor sao por organizacao.")

## 4 · Os autotestes, que não gastam token

**Rode isto antes de qualquer coisa que gaste cota.** Leva menos de um segundo e pega o
erro mais caro do projeto: o mundo não conter a causa dos casos.

É o sucessor do `verificar_notebook.py` nesta fase: a mesma ideia, agora como script.

In [ ]:
!python testes.py

## 5 · O teste de memória

Três chamados — **C-103, C-106 e C-108** — com **a mesma causa**: a revisão de perfis de
-6h removeu o grupo `erp_leitura` de 12 usuários, e os três estão entre eles.

Quem já apurou isso no primeiro não precisa apurar de novo. Quem não lembra, **refaz as
quatro consultas três vezes**.

A célula roda os três de dois jeitos, **sem memória** e **com memória**, e põe a conta
lado a lado. Leva uns 3 a 4 minutos, por causa das esperas de 20 s entre chamados.

In [ ]:
!python rodar_memoria.py

### Leia a tabela antes de seguir

Três perguntas, e a célula já as imprime:

1. quantas consultas caíram?
2. quanto o custo caiu?
3. **a entrada cresceu quanto?**

**A terceira explica a segunda.** Memória é contexto carregado: cada chamado seguinte leva
a conversa inteira junto, então a entrada cresce. Se as consultas caírem muito e o custo
cair pouco, você acabou de medir **o preço da memória**.

E confira a coluna `causa` nas duas. **Economizar consulta só vale se a resposta continuar
certa.**

## 6 · O mesmo agente, agora no *framework*

Mesmo mundo, mesmas seis consultas, mesma instrução. **Só a orquestração muda.**

Se esta célula falhar, não é problema seu: siga com o resultado da anterior. O conteúdo do
bloco está lá.

In [ ]:
!python rodar_memoria.py --framework

## 7 · O que sumiu, e é o ponto da aula

A célula abaixo mostra **o mesmo trecho** dos dois arquivos, lado a lado. Não gasta token.

In [ ]:
import re, pathlib

def mostrar(caminho, titulo, inicio, linhas=18):
    texto = pathlib.Path(caminho).read_text(encoding="utf-8")
    i = texto.index(inicio)
    trecho = texto[i:].splitlines()[:linhas]
    print("=" * 74)
    print(titulo)
    print("=" * 74)
    print("\n".join(trecho))
    print()

mostrar("vertice/artesanal.py",
        "ARTESANAL -- declarar UMA ferramenta", "_D_USUARIO =", 12)
mostrar("vertice/framework.py",
        "FRAMEWORK -- declarar AS SEIS", "FERRAMENTAS = [", 10)

print("=" * 74)
print("A ferramenta virou a FUNCAO, e a description virou a DOCSTRING.")
print("Todo o criterio da clinica do Encontro 4 continua valendo, no mesmo")
print("lugar de sempre. O framework tira de voce o SCHEMA, nao a decisao.")
print()
print("E o laco de 30 linhas do Encontro 3 virou uma chamada:")
print("    agente.run(pedido, session=sessao)")
print()
print("E a mesma troca que voces mediram no Encontro 9 com o LangChain:")
print("menos linhas, e as decisoes escondidas atras de padroes.")

## 8 · Em equipe, no papel

> **O seu projeto já tem algum destes três problemas?**
>
> retomar depois de falha · grafo com ramos · observar execuções passadas
>
> Escreva **qual**, e a evidência. Se não tiver nenhum, escreva **"nenhum"**.

**"Nenhum" é resposta certa quando for verdade**, do mesmo jeito que *"o nosso caso é
workflow"* era resposta certa no Marco 1.

## 9 · A tarefa: sair do Colab

Este notebook rodou tudo. O que ele **não** faz é `commit` e `push`, e isso você vai
precisar antes do Marco 1, que é entregue por ***tag*** no repositório.

**O passo a passo completo está no `GUIA_VSCODE.md`**, escrito para ser seguido sem
pressa: instalar, autenticar, clonar, rodar e empurrar. Leva cerca de 40 minutos na
primeira vez, e o plantão existe para isso.

> **Por que vale a pena, e não é burocracia:** a partir daqui o projeto da sua equipe é um
> repositório, não uma pasta de notebooks. É o que torna possível duas pessoas mexerem no
> mesmo código, e é o que o Marco 2 vai pedir com *branch* e *pull request*.